In [2]:
from constant import openalex_path
import polars as pl

source_lf = pl.scan_parquet(f"{openalex_path}/works/primary_location_id/*")
year_lf = pl.scan_parquet(f"{openalex_path}/works/publication_year/*")
ref_lf = pl.scan_parquet(f"{openalex_path}/works/referenced_works/*")

In [3]:
citing_preprints_df = pl.read_parquet("./data/citing_preprints_full.parquet")

In [3]:
from constant import inhouse_titles_v1
inhouse_titles = inhouse_titles_v1

source_name_lf = pl.scan_parquet(f"{openalex_path}/sources/display_name/*").with_columns(pl.col("display_name").str.to_lowercase().alias("source_name"))
inhouse_source_ids = (
    source_name_lf
    .filter(
        pl.col("source_name").is_in([t.lower() for t in inhouse_titles])
    )
    .collect(engine="streaming")
)

inhouse_source_ids.write_csv("./data/inhouse_source_ids.csv")


/tmp/ipykernel_46900/368614740.py:10: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.
  .collect(engine="streaming")


%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Manual Inspection of the file...

%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

In [4]:
inhouse_source_ids = pl.read_csv("./data/inhouse_source_ids_inspected.csv")
CNS_journals = inhouse_source_ids["source_id"].to_list() # need manual inspection

ref_count_lf = (
    ref_lf
    .with_columns(
        pl.col("referenced_works").list.len().alias("ref_count")
    )
    .select("work_id", "ref_count")
)

In [5]:
CNS_papers_df = (
    source_lf
    .join(year_lf, on="work_id", how="inner")
    .join(ref_count_lf, on="work_id", how="left")
    .filter(
        pl.col("primary_location_id").is_in(CNS_journals),
    )
    .collect(engine="streaming")
)
CNS_papers_df.write_parquet("./data/CNS_paper_details.parquet")

/tmp/ipykernel_48083/3377837990.py:8: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.
  .collect(engine="streaming")


In [6]:
CNS_citing_preprints_df = (
    citing_preprints_df
    .join(CNS_papers_df.select("work_id"), on="work_id", how="semi")
    .lazy()
    .join(year_lf, left_on="referenced_works", right_on="work_id", how="left", suffix="_cited")
    .collect(engine="streaming")
)

CNS_citing_preprints_df.write_parquet("./data/CNS_citing_preprints.parquet")
CNS_citing_preprints_df.head()

work_id,referenced_works,doi,primary_category,server,curated_doi,primary_location_id,publication_year,publication_year_cited
str,str,str,str,str,str,str,i64,i64
"""W4409615778""","""W4404595734""","""10.1101/2024.11.19.624383""","""neuroscience""","""biorxiv""",null,"""S169236886""",2025,2024
"""W4309632154""","""W3164879361""","""10.1101/2020.12.07.414151""","""genomics""","""biorxiv""","""10.3389/fimmu.2021.768189""","""S2898612692""",2022,2020
"""W4409283340""","""W4403545558""","""10.1101/2024.10.16.618682""","""plant biology""","""biorxiv""",null,"""S128425624""",2025,2024
"""W4407855241""","""W4392011893""","""10.48550/arxiv.2402.11025""","""cs.LG""","""arxiv""",null,"""S2898612692""",2025,2024
"""W4407855241""","""W4392011893""","""10.48550/arxiv.2402.11025""","""cs.LG""","""arxiv""",null,"""S2898612692""",2025,2024


In [12]:
period = [
    (1800, 2000, "Before 2000"),
    (2000, 2015, "2000-2015"),
    (2015, 2020, "2015-2019"),
]

breaks = [p[1] for p in period[:-1]]
labels = [p[2] for p in period]

CNS_journals_yearly_count = (
    CNS_papers_df
    .group_by("primary_location_id", "publication_year")
    .len()
)

CNS_journals_inception = (
    CNS_journals_yearly_count
    .filter(pl.col("len") > 5)
    .sort("publication_year", descending=False)
    .group_by("primary_location_id")
    .head(1)
    .select(
        pl.col("primary_location_id"),
        pl.col("publication_year").alias("journal_inception")
    )
    .sort("journal_inception")
)

journal_detail = (
    CNS_journals_inception
    .select("primary_location_id", "journal_inception")
    .filter(pl.col("journal_inception").is_between(1800, 2020))
    .with_columns(
        pl.col("journal_inception").cut(breaks=breaks, labels=labels).alias("inception_period")
    )
)

journal_detail.write_csv("./data/CNS_journal_inception.parquet")
# manual inspection

In [8]:
CNS_citing_preprints_df.head()
# work_id	referenced_works	doi	primary_category	server	curated_doi	primary_location_id	publication_year	journal_inception	inception_period	publication_year_cited
# str	str	str	str	str	str	str	i64	i64	cat	i64
# "W4409615778"	"W4404595734"	"10.1101/2024.11.19.624383"	"neuroscience"	"biorxiv"	null	"S169236886"	2025	2012	"2000-2015"	2024
# "W4409283340"	"W4403545558"	"10.1101/2024.10.16.618682"	"plant biology"	"biorxiv"	null	"S128425624"	2025	1991	"Before 2000"	2024
# "W4309632154"	"W3164879361"	"10.1101/2020.12.07.414151"	"genomics"	"biorxiv"	"10.3389/fimmu.2021.768189"	"S2898612692"	2022	2015	"2000-2015"	2020
# "W4312081285"	"W4295188023"	"10.48550/arxiv.2011.11869"	"cond-mat.mtrl-sci"	"arxiv"	null	"S2912241403"	2022	2018	"2015-2019"	2020
# "W4407855241"	"W4392011893"	"10.48550/arxiv.2402.11025"	"cs.LG"	"arxiv"	null	"S2898612692"	2025	2015	"2000-2015"	2024


work_id,referenced_works,doi,primary_category,server,curated_doi,primary_location_id,publication_year,publication_year_cited
str,str,str,str,str,str,str,i64,i64
"""W4409615778""","""W4404595734""","""10.1101/2024.11.19.624383""","""neuroscience""","""biorxiv""",null,"""S169236886""",2025,2024
"""W4309632154""","""W3164879361""","""10.1101/2020.12.07.414151""","""genomics""","""biorxiv""","""10.3389/fimmu.2021.768189""","""S2898612692""",2022,2020
"""W4409283340""","""W4403545558""","""10.1101/2024.10.16.618682""","""plant biology""","""biorxiv""",null,"""S128425624""",2025,2024
"""W4407855241""","""W4392011893""","""10.48550/arxiv.2402.11025""","""cs.LG""","""arxiv""",null,"""S2898612692""",2025,2024
"""W4407855241""","""W4392011893""","""10.48550/arxiv.2402.11025""","""cs.LG""","""arxiv""",null,"""S2898612692""",2025,2024


In [9]:
CNS_citing_preprints_df.select("publication_year").to_series().hist()

breakpoint,category,count
f64,cat,u32
1986.3,"""[1982.0, 1986.3]""",2
1990.6,"""(1986.3, 1990.6]""",3
1994.9,"""(1990.6, 1994.9]""",2
1999.2,"""(1994.9, 1999.2]""",68
2003.5,"""(1999.2, 2003.5]""",155
2007.8,"""(2003.5, 2007.8]""",157
2012.1,"""(2007.8, 2012.1]""",391
2016.4,"""(2012.1, 2016.4]""",1261
2020.7,"""(2016.4, 2020.7]""",13213


In [10]:
(
    CNS_citing_preprints_df
    .filter(
        pl.col("publication_year") >= pl.col("publication_year_cited"),
        pl.col("primary_category").str.contains("bio")
        # pl.col("primary_category") != "astro-ph"
    )
    .sort("publication_year")
    .head(20)
)

work_id,referenced_works,doi,primary_category,server,curated_doi,primary_location_id,publication_year,publication_year_cited
str,str,str,str,str,str,str,i64,i64
"""W2156050324""","""W2953302082""","""10.48550/arxiv.physics/0208057""","""physics.bio-ph""","""arxiv""",null,"""S45757444""",2003,2002
"""W1531414226""","""W72248065""","""10.48550/arxiv.q-bio/0312021""","""q-bio.MN""","""arxiv""",null,"""S8553189""",2004,2003
"""W2163480486""","""W2953138302""","""10.48550/arxiv.q-bio/0310024""","""q-bio.GN""","""arxiv""",null,"""S8553189""",2004,2003
"""W1531414226""","""W2964351603""","""10.48550/arxiv.q-bio/0312022""","""q-bio.MN""","""arxiv""",null,"""S8553189""",2004,2003
"""W2169720294""","""W1675944701""","""10.48550/arxiv.q-bio/0505003""","""q-bio.NC""","""arxiv""",null,"""S45757444""",2005,2005
…,…,…,…,…,…,…,…,…
"""W1982583124""","""W1658329450""","""10.48550/arxiv.1207.2484""","""q-bio.QM""","""arxiv""",null,"""S106963461""",2012,2012
"""W2043914340""","""W1572506765""","""10.48550/arxiv.0805.2249""","""physics.bio-ph""","""arxiv""",null,"""S137773608""",2012,2008
"""W2161350677""","""W1596062789""","""10.48550/arxiv.1207.3907""","""q-bio.GN""","""arxiv""",null,"""S110447773""",2013,2012


In [11]:
# set the polars display options set the truncate length
pl.Config.set_fmt_str_lengths(100)
doi_lf = pl.scan_parquet(f"{openalex_path}/works/doi/*")

citing_samples = (
    CNS_citing_preprints_df
    .filter(
        pl.col("publication_year") <=  2003
    )
    .lazy()
    .join(doi_lf, on="work_id", how="left")
    .sort("publication_year")
    .sink_csv("./data/citing_samples.csv")
)

# Cluster papers by SciBERT

In [34]:
import polars as pl
CNS_paper_details = pl.read_parquet("./data/CNS_paper_details.parquet")
journal_detail = pl.read_parquet("./data/CNS_journal_inception.parquet")

CNS_papers_df = (
    CNS_paper_details
    .join(journal_detail, on="primary_location_id", how="left")
    .filter(pl.col("inception_period") == "Before 2000")
)
CNS_papers_df.height

990489

In [35]:
abstract_lf = pl.scan_parquet(f"{openalex_path}/works/abstract/*")

CNS_papers_abs = (
    CNS_papers_df
    .lazy().join(abstract_lf, on="work_id", how="left")
    .collect(engine="streaming")
)


In [36]:
CNS_papers_abs.with_columns(
    pl.col("abstract").is_null().alias("abstract_missing")
).group_by("abstract_missing").len()

abstract_missing,len
bool,u32
false,13810
true,976679


In [37]:
CNS_papers_abs = CNS_papers_abs.filter(~pl.col("abstract").is_null())

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available())

2.9.1+cpu False


In [ ]:
from transformers import AutoTokenizer, AutoModel

model = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model)
mdl = AutoModel.from_pretrained(model)

print("Loaded SciBERT!")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loaded SciBERT!


In [38]:
# Embed one sample
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

# --- SciBERT ---
model_name = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def embed(text):
    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model(**encoded)

    # mean pooling
    emb = output.last_hidden_state.mean(dim=1).squeeze()
    return emb.cpu().numpy()

# --- サンプル abstract ---
text = "We present a method for analyzing citation behavior in high-impact scientific journals."

vec = embed(text)
print(vec.shape)
print(vec)

(768,)
[-4.74826187e-01 -3.70428801e-01  5.66542670e-02 -1.04749918e-01
 -1.59150586e-01 -2.85923891e-02  1.14607148e-01  7.00981319e-01
 -6.83914483e-01 -1.01674736e-01  1.92069963e-01 -2.76718307e-02
 -3.34080517e-01 -4.47009414e-01 -6.55444443e-01  5.07011890e-01
  3.87857616e-01 -1.08410798e-01  3.13989222e-01  1.75675869e-01
  1.09991431e-01  1.01042408e-02  2.67497957e-01 -3.25993925e-01
  5.17110467e-01 -4.86944258e-01  2.18043461e-01 -2.15770200e-01
 -3.70355427e-01 -8.41331005e-01 -8.18293775e-04 -4.83175963e-01
 -4.26585406e-01 -1.02452254e+00  1.78077877e-01 -1.26354337e-01
  3.28308016e-01  1.04716241e-01 -4.88759488e-01  6.80985391e-01
 -8.67063224e-01  5.05873621e-01  6.78104281e-01 -4.33158100e-01
  4.82743606e-02  2.50844777e-01  5.43750763e-01  7.18353510e-01
  1.51230246e-01 -6.91979289e-01  4.98883575e-01  7.26993680e-01
  6.91674650e-01  3.04355532e-01  3.42413902e-01 -3.50212932e-01
 -5.40212095e-01 -1.66298077e-01  2.63583690e-01  1.98781807e-02
 -4.91345644e-01  

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np

# --- 1. モデルのロード ---
# SciBERT (AllenAI) の英語科学文献向けモデル
MODEL_NAME = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

# GPU があれば使う
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- 2. 1つの文章をベクトル化する関数 ---
def embed_text(text: str):
    if not isinstance(text, str) or text.strip() == "":
        # 空や欠損にはゼロベクトルを返す
        return np.zeros(model.config.hidden_size)

    encoded = tokenizer(
        text,
        padding=True,
        truncation=True,        # 長文は512 tokens で切る
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model(**encoded)

    # ベクトル化の方法はいくつかあるが、ここでは平均プーリングが安定
    # CLS だけ使う場合 → output.last_hidden_state[:, 0, :]
    embedding = output.last_hidden_state.mean(dim=1).squeeze()

    return embedding.cpu().numpy()

# --- 3. DataFrame 全体をベクトル化 ---
# CNS_papers_df["Abstract"] をベクトル化
embeddings = []
from tqdm import tqdm
for abstract in tqdm(CNS_papers_abs["abstract"]):
    emb = embed_text(abstract)
    embeddings.append(emb)
    if len(embeddings) % 100 == 0:
        np.save(f"./data/temp_embedding.npy", np.vstack(embeddings))

# numpy array に変換 (N, 768)
embeddings = np.vstack(embeddings)

# DataFrameに追加 (optional)
for i in range(embeddings.shape[1]):
    CNS_papers_abs[f"embed_{i}"] = embeddings[:, i]

 18%|█▋       | 2536/13810 [27:39<1:54:52,  1.64it/s]